In [ ]:
from collections import defaultdict
from typing import List, Dict, Any

import json
import os

from natasha import (
    Segmenter, MorphVocab,
    NewsEmbedding, NewsMorphTagger, NewsSyntaxParser, NewsNERTagger,
    Doc
)
from datasets import load_dataset

from src.parser import process_dataset
from src.utils import load_natasha_data, get_stats, analyze_entity_patterns
from src.eval import evaluate, get_error_pairs


VERBOSE_FLAG = False

In [2]:
dataset = load_dataset("gusevski/factrueval2016", cache_dir='data')
dataset

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['data'],
        num_rows: 1
    })
    validation: Dataset({
        features: ['data'],
        num_rows: 1
    })
    test: Dataset({
        features: ['data'],
        num_rows: 1
    })
})

In [3]:
entity_counts = get_stats(dataset) # type: ignore
entity_counts

{'train': {'PER': 10284, 'LOC': 5307, 'ORG': 8311, 'O': 132788},
 'validation': {'PER': 3431, 'LOC': 1701, 'ORG': 2815, 'O': 43239},
 'test': {'PER': 3517, 'LOC': 1850, 'ORG': 2824, 'O': 44847}}

In [4]:
entity_counts, natasha_data = process_dataset("gusevski/factrueval2016")

if VERBOSE_FLAG:
    analyze_entity_patterns(entity_counts)

    for split in ['train', 'validation', 'test']:
        total_entities = sum(len(entity_counts[split][et]) for et in ['PER', 'LOC', 'ORG'])
        total_o = len(entity_counts[split]['O'])
        
        print(f"\n{split.upper()}:")
        print(f"  Всего сущностей: {total_entities}")
        print(f"  PER: {len(entity_counts[split]['PER'])}")
        print(f"  LOC: {len(entity_counts[split]['LOC'])}") 
        print(f"  ORG: {len(entity_counts[split]['ORG'])}")
        print(f"  O-токенов: {total_o}")
        print(f"  Соотношение сущности/O: {total_entities/(total_entities + total_o):.3f}")

Repo card metadata block was not found. Setting CardData to empty.


### NATASHA

In [5]:
class NatashaPipeline:
    def __init__(self):

        self.segmenter = Segmenter()
        self.morph_vocab = MorphVocab()
        
        self.emb = NewsEmbedding()
        self.morph_tagger = NewsMorphTagger(self.emb)
        self.syntax_parser = NewsSyntaxParser(self.emb) 
        self.ner_tagger = NewsNERTagger(self.emb)
    
    
    def process_text(self, text: str) -> Doc:
       
        doc = Doc(text)
        
        doc.segment(self.segmenter)
        doc.tag_morph(self.morph_tagger)
        doc.parse_syntax(self.syntax_parser) 
        doc.tag_ner(self.ner_tagger)
        
        for span in doc.spans:
            span.normalize(self.morph_vocab)
            
        return doc
    
    
    def extract_entities(self, doc: Doc) -> List[Dict[str, Any]]:
        
        entities = []
        
        for span in doc.spans:
            if span.type in ['PER', 'LOC', 'ORG']:
                entity = {
                    'text': span.text,
                    'type': span.type,
                    'start': span.start,
                    'end': span.stop,
                    'normalized': span.normal
                }
                entities.append(entity)
        
        return entities
    
    
    def inference(self, natasha_data: List) -> List:

        results = []
        
        for _, example in enumerate(natasha_data):
            
            text = example['text']
            true_entities = example['entities']            

            doc = self.process_text(text)
            predicted_entities = self.extract_entities(doc)

            results.append({
                'text': text,
                'true_entities': true_entities,
                'predicted_entities': predicted_entities,
                'id': example['id']
            })
        
        return results

In [6]:
natasha_data = load_natasha_data()

pipeline = NatashaPipeline()

Загружено 7746 примеров из train
Загружено 2582 примеров из validation
Загружено 2582 примеров из test


/home/timmiakov/repos/hse-materials/.venv/lib/python3.13/site-packages/pymorphy2/analyzer.py:114: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### Test on one sample

In [7]:
test_example = natasha_data['validation'][0]
doc = pipeline.process_text(test_example['text'])
entities = pipeline.extract_entities(doc)

print(f"Текст: {test_example['text'][:100]}...")
print("\nИстинные сущности:")
for ent in test_example['entities']:
    print(f"  {ent['type']}: '{ent['text']}'")

print("\nНайденные Natasha сущности:")
for ent in entities:
    print(f"  {ent['type']}: '{ent['text']}' (нормализовано: '{ent['normalized']}')")

Текст: как акционерный коммерческий Московский муниципальный банк - Банк Москвы ,контрольный пакет акций ко...

Истинные сущности:
  ORG: 'Московский муниципальный банк - Банк Москвы'
  LOC: 'Москвы'

Найденные Natasha сущности:
  ORG: 'Московский муниципальный банк' (нормализовано: 'Московский муниципальный банк')
  ORG: 'Банк Москвы' (нормализовано: 'Банк Москвы')
  LOC: 'Москвы' (нормализовано: 'Москва')


### EXP1: Baseline Natasha

In [8]:
results = pipeline.inference(natasha_data['test'])

In [9]:
evaluate(results)

Общие метрики:
  Precision: 0.919
  Recall:    0.874
  F1:        0.896
  True entities: 5486
  Predicted entities: 5217
  Matches: 4795

Детализация по типам:
  PER:
    Precision: 0.958
    Recall:    0.900
    F1:        0.928
    True: 2187, Predicted: 2056, Matches: 1969
  LOC:
    Precision: 0.958
    Recall:    0.976
    F1:        0.967
    True: 1509, Predicted: 1538, Matches: 1473
  ORG:
    Precision: 0.834
    Recall:    0.756
    F1:        0.793
    True: 1790, Predicted: 1623, Matches: 1353


In [20]:
error_pairs = get_error_pairs(results)

In [25]:
false_postitive_pairs = [(e['false_negatives'], e['predicted_entities']) for e in error_pairs if e['false_negatives']]
false_postitive_pairs

[([{'text': 'Боксер', 'type': 'PER', 'start': 0, 'end': 6},
   {'text': 'Карли Фиорину', 'type': 'PER', 'start': 16, 'end': 29},
   {'text': 'Carly', 'type': 'PER', 'start': 31, 'end': 36}],
  [{'text': 'Карли Фиорину( Carly Fiorina )',
    'type': 'PER',
    'start': 16,
    'end': 46,
    'normalized': 'Карли Фиорину( Carly Fiorina )'},
   {'text': 'Hewlett-Packard',
    'type': 'ORG',
    'start': 88,
    'end': 103,
    'normalized': 'Hewlett-Packard'}]),
 ([{'text': 'К-Агро', 'type': 'ORG', 'start': 58, 'end': 64},
   {'text': 'ОАО', 'type': 'ORG', 'start': 91, 'end': 94},
   {'text': 'ОАО', 'type': 'ORG', 'start': 203, 'end': 206},
   {'text': 'Корпорация развития Калужской',
    'type': 'ORG',
    'start': 210,
    'end': 239},
   {'text': 'К-Агро', 'type': 'ORG', 'start': 403, 'end': 409}],
  [{'text': 'Калужской области',
    'type': 'LOC',
    'start': 2,
    'end': 19,
    'normalized': 'Калужская область'},
   {'text': "ОАО `` Россельхозбанк '' I-ORG",
    'type': 'ORG',
  

### EXP2: Natasha